# E-commerce Sales & Profitability Analysis
### Data Analyst Portfolio Project

**Objective:** Analyze sales performance, product profitability, customer value, and potential cross-selling opportunities using a synthetic e-commerce dataset.

**Tools:** Python, pandas, NumPy, Matplotlib, Seaborn.

**Important:** The dataset is synthetic. Results demonstrate an analytical workflow; they are not evidence of real-company performance or causal campaign impact.

## 1. Setup and data loading

Run this notebook from the repository root. If you open the notebook in Google Colab, upload the CSV or update `DATA_PATH` to its mounted location.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("data/ecommerce_portfolio_dataset_15000.csv")
if not DATA_PATH.exists():
    # Fallback for running the notebook beside the CSV
    DATA_PATH = Path("ecommerce_portfolio_dataset_15000.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["order_date"])
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Date range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")
df.head()

## 2. Data quality checks

Check duplicates, missing values, and basic schema. Missing values are not automatically filled with zero: zero has a business meaning and should not be used as a generic missing-value replacement.

In [ ]:
quality_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
}).sort_values("missing_count", ascending=False)

display(quality_summary)
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Unique order IDs:", df["order_id"].nunique())
print("Unique customer IDs:", df["customer_id"].nunique())

In [ ]:
# Standardize types without changing business values
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["order_month"] = df["order_date"].dt.to_period("M")

numeric_cols = [
    "quantity", "unit_price_inr", "discount_pct", "gross_sales_inr",
    "discount_amount_inr", "net_sales_inr", "unit_cost_inr",
    "profit_inr", "shipping_days", "customer_rating", "marketing_spend_inr"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

categorical_cols = [
    "customer_type", "city", "state", "sales_channel",
    "product_category", "product_name", "payment_method", "order_status"
]
for col in categorical_cols:
    df[col] = df[col].astype("category")

# Validate arithmetic where source values are available
expected_gross = df["quantity"] * df["unit_price_inr"]
gross_mismatch = ~np.isclose(
    df["gross_sales_inr"], expected_gross, atol=0.02, equal_nan=True
)
expected_net = df["gross_sales_inr"] - df["discount_amount_inr"]
net_mismatch = ~np.isclose(
    df["net_sales_inr"], expected_net, atol=0.02, equal_nan=True
)

print("Gross sales arithmetic mismatches:", int(gross_mismatch.sum()))
print("Net sales arithmetic mismatches:", int(net_mismatch.sum()))
print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

## 3. KPI overview

Use `net_sales_inr` for net sales and AOV. The original notebook mistakenly used `gross_sales_inr` while labelling it as net sales. Financial results below include all rows unless an explicit filter is applied; inspect `order_status` and confirm the business definition before interpreting them as realized revenue.

In [ ]:
total_orders = df["order_id"].nunique()
total_customers = df["customer_id"].nunique()
net_sales = df["net_sales_inr"].sum()
estimated_profit = df["profit_inr"].sum()
units_sold = df["quantity"].sum()

kpis = pd.DataFrame({
    "Metric": [
        "Orders", "Customers", "Net sales (INR)",
        "Estimated profit (INR)", "Estimated profit margin (%)",
        "Average order value (INR)", "Units"
    ],
    "Value": [
        total_orders, total_customers, net_sales, estimated_profit,
        (estimated_profit / net_sales * 100) if net_sales else np.nan,
        (net_sales / total_orders) if total_orders else np.nan,
        units_sold
    ]
})
display(kpis)

## 4. Category performance

Compare sales, estimated profit, and profit margin together. A category with the most sales is not necessarily the most profitable.

In [ ]:
category_analysis = (
    df.groupby("product_category", observed=True)
      .agg(
          orders=("order_id", "nunique"),
          net_sales_inr=("net_sales_inr", "sum"),
          estimated_profit_inr=("profit_inr", "sum"),
          units=("quantity", "sum")
      )
      .reset_index()
)
category_analysis["profit_margin_pct"] = (
    category_analysis["estimated_profit_inr"] /
    category_analysis["net_sales_inr"].replace(0, np.nan) * 100
)
category_analysis = category_analysis.sort_values("net_sales_inr", ascending=False)
display(category_analysis.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
categories = category_analysis["product_category"].astype(str)
axes[0].bar(categories, category_analysis["net_sales_inr"])
axes[0].set_title("Net Sales by Category")
axes[0].set_xlabel("")
axes[0].set_ylabel("Net sales (INR)")
axes[0].tick_params(axis="x", rotation=35)

axes[1].bar(categories, category_analysis["profit_margin_pct"])
axes[1].set_title("Estimated Profit Margin by Category")
axes[1].set_xlabel("")
axes[1].set_ylabel("Profit margin (%)")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 5. Monthly performance and January–February 2025 comparison

In [ ]:
monthly = (
    df.groupby("order_month", observed=True)
      .agg(
          net_sales_inr=("net_sales_inr", "sum"),
          estimated_profit_inr=("profit_inr", "sum"),
          orders=("order_id", "nunique")
      )
      .reset_index()
      .sort_values("order_month")
)
monthly["aov_inr"] = monthly["net_sales_inr"] / monthly["orders"].replace(0, np.nan)
monthly["month"] = monthly["order_month"].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
sns.lineplot(data=monthly, x="month", y="net_sales_inr", marker="o", ax=ax)
ax.set_title("Monthly Net Sales")
ax.set_xlabel("Month")
ax.set_ylabel("Net sales (INR)")
ax.tick_params(axis="x", rotation=70)
plt.tight_layout()
plt.show()

jan_feb = monthly[monthly["month"].isin(["2025-01", "2025-02"])].set_index("month")
display(jan_feb[["net_sales_inr", "estimated_profit_inr", "orders", "aov_inr"]].round(2))

if {"2025-01", "2025-02"}.issubset(jan_feb.index):
    for metric in ["net_sales_inr", "estimated_profit_inr", "orders", "aov_inr"]:
        jan_value = jan_feb.loc["2025-01", metric]
        feb_value = jan_feb.loc["2025-02", metric]
        pct = (feb_value / jan_value - 1) * 100 if jan_value else np.nan
        print(f"{metric}: {pct:.2f}% month-over-month")

## 6. Customer value segmentation

Segments are defined using customer-level net sales tertiles. They are relative value groups, not validated lifecycle labels. Do not treat the `customer_type` column as reliable lifecycle evidence unless its generation method is known and validated.

In [ ]:
customer_value = (
    df.groupby("customer_id", observed=True)
      .agg(
          total_orders=("order_id", "nunique"),
          total_revenue_inr=("net_sales_inr", "sum"),
          total_profit_inr=("profit_inr", "sum")
      )
      .reset_index()
)
customer_value["customer_segment"] = pd.qcut(
    customer_value["total_revenue_inr"].rank(method="first"),
    q=3,
    labels=["Low Value", "Medium Value", "High Value"]
)

segment_analysis = (
    customer_value.groupby("customer_segment", observed=True)
      .agg(
          customers=("customer_id", "nunique"),
          revenue_inr=("total_revenue_inr", "sum"),
          profit_inr=("total_profit_inr", "sum"),
          average_orders=("total_orders", "mean")
      )
      .reset_index()
)
segment_analysis["revenue_share_pct"] = (
    segment_analysis["revenue_inr"] / segment_analysis["revenue_inr"].sum() * 100
)
segment_analysis["profit_margin_pct"] = (
    segment_analysis["profit_inr"] / segment_analysis["revenue_inr"].replace(0, np.nan) * 100
)
display(segment_analysis.round(2))

df_segmented = df.merge(
    customer_value[["customer_id", "customer_segment"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)
assert len(df_segmented) == len(df), "Segment merge changed the row count."

In [ ]:
segment_category = (
    df_segmented.groupby(["customer_segment", "product_category"], observed=True)
      .agg(
          net_sales_inr=("net_sales_inr", "sum"),
          estimated_profit_inr=("profit_inr", "sum"),
          orders=("order_id", "nunique")
      )
      .reset_index()
)
segment_category["profit_margin_pct"] = (
    segment_category["estimated_profit_inr"] /
    segment_category["net_sales_inr"].replace(0, np.nan) * 100
)
display(segment_category.round(2))

## 7. High-value customer Fashion trend

This section uses the **High Value segment defined from full-period customer revenue**. Because the segment is based on the full dataset, it is descriptive rather than a segment assignment that would have been known at the beginning of 2024.

In [ ]:
fashion_high = df_segmented[
    (df_segmented["customer_segment"] == "High Value") &
    (df_segmented["product_category"] == "Fashion")
].copy()
fashion_high["year"] = fashion_high["order_date"].dt.year

fashion_annual = (
    fashion_high.groupby("year")
      .agg(
          revenue_inr=("net_sales_inr", "sum"),
          profit_inr=("profit_inr", "sum"),
          orders=("order_id", "nunique"),
          customers=("customer_id", "nunique")
      )
)
fashion_annual["aov_inr"] = fashion_annual["revenue_inr"] / fashion_annual["orders"]
fashion_annual["profit_margin_pct"] = (
    fashion_annual["profit_inr"] / fashion_annual["revenue_inr"] * 100
)
display(fashion_annual.round(2))

In [ ]:
fashion_monthly = (
    fashion_high.assign(month=fashion_high["order_date"].dt.to_period("M").astype(str))
      .groupby("month")
      .agg(
          revenue_inr=("net_sales_inr", "sum"),
          profit_inr=("profit_inr", "sum"),
          orders=("order_id", "nunique"),
          customers=("customer_id", "nunique")
      )
      .reset_index()
      .sort_values("month")
)
fig, ax = plt.subplots(figsize=(14, 5))
sns.lineplot(data=fashion_monthly, x="month", y="revenue_inr", marker="o", ax=ax)
ax.set_title("Fashion Revenue: High-Value Segment")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (INR)")
ax.tick_params(axis="x", rotation=70)
plt.tight_layout()
plt.show()

## 8. Cross-selling eligibility and experiment design

Eligibility definition for this proposed experiment:
- Purchased Fashion in 2024.
- Did not purchase Fashion in 2025.
- Purchased Beauty in 2025.
- Has a valid customer ID.

This notebook identifies the eligible population only. It does not claim the campaign was run or that incremental profit was generated. Rebuild eligibility using current transactions before any real 2026 campaign.

In [ ]:
fashion_2024_ids = set(df.loc[
    (df["product_category"] == "Fashion") & (df["order_date"].dt.year == 2024),
    "customer_id"
].dropna())

fashion_2025_ids = set(df.loc[
    (df["product_category"] == "Fashion") & (df["order_date"].dt.year == 2025),
    "customer_id"
].dropna())

beauty_2025_ids = set(df.loc[
    (df["product_category"] == "Beauty") & (df["order_date"].dt.year == 2025),
    "customer_id"
].dropna())

eligible_customers = sorted((fashion_2024_ids - fashion_2025_ids) & beauty_2025_ids)
print("Eligible customers (all customer-value segments):", len(eligible_customers))

### Proposed A/B test

- **Control:** business-as-usual communications.
- **Treatment:** business-as-usual plus a personalized Fashion offer.
- **Randomization unit:** customer; keep assignment fixed.
- **Primary metric:** net profit per assigned customer over a fixed 30-day period, net of campaign costs where available.
- **Guardrail:** total profit across all categories, to detect cannibalization.
- **Secondary metrics:** Fashion purchase rate, Fashion revenue per assigned customer, discount cost.
- **Analysis:** intention-to-treat comparison; report effect size and confidence interval.
- **Limitations:** the available historical dataset ends in 2025; assignments created now cannot be used to claim historical causal impact. The eligible cohort is small, so estimate power with a larger comparable historical sample before launch.

In [ ]:
# Create a reproducible assignment for planning/testing the workflow only.
# Do not regenerate assignments after a real campaign begins; persist them.
rng = np.random.default_rng(42)
shuffled = np.array(eligible_customers, dtype=object)
rng.shuffle(shuffled)
n_control = len(shuffled) // 2

assignments = pd.DataFrame({
    "customer_id": shuffled,
    "experiment_group": [
        "control" if i < n_control else "treatment"
        for i in range(len(shuffled))
    ]
})
display(assignments["experiment_group"].value_counts().to_frame("customers"))

# Validation
eligibility_check = pd.DataFrame({
    "customer_id": assignments["customer_id"],
    "fashion_2024": assignments["customer_id"].isin(fashion_2024_ids),
    "fashion_2025": assignments["customer_id"].isin(fashion_2025_ids),
    "beauty_2025": assignments["customer_id"].isin(beauty_2025_ids)
})
assert eligibility_check["fashion_2024"].all()
assert (~eligibility_check["fashion_2025"]).all()
assert eligibility_check["beauty_2025"].all()
print("Eligibility checks passed.")

## 9. Business recommendations

1. Investigate the drivers of monthly sales changes by product, channel, and order volume before attributing causes.
2. Compare category revenue with estimated profit margin; high revenue does not guarantee high profitability.
3. Use customer segments as descriptive groups and check whether results are robust to alternative definitions.
4. Validate cost coverage in `profit_inr` before treating it as contribution profit.
5. Run a prospective randomized cross-sell experiment only after refreshing the eligible customer list, checking contact consent, estimating sample size, and defining campaign costs.

## 10. Limitations

- The data is synthetic and does not establish real-world business outcomes.
- The `profit_inr` field may exclude overhead, shipping, returns, payment fees, or campaign costs; treat it as estimated profit unless the data dictionary confirms full cost coverage.
- The dataset contains cancelled and returned statuses. Financial reporting should explicitly define whether those records are included or excluded; this notebook shows all-row analysis unless stated otherwise.
- Customer segments are defined using full-period revenue, so they should not be interpreted as historical real-time labels.
- The A/B test is a proposal. Historical comparisons are not causal evidence of campaign effectiveness.
- Marketing spend should not be used to claim ROAS unless spend is accurately attributed to the relevant campaign and time window.

## Appendix: save key outputs

Uncomment the lines below if you want to export summary tables for a report or dashboard.

In [ ]:
# category_analysis.to_csv("category_analysis.csv", index=False)
# monthly.to_csv("monthly_kpis.csv", index=False)
# segment_analysis.to_csv("customer_segment_analysis.csv", index=False)
# fashion_annual.to_csv("high_value_fashion_annual.csv")
# assignments.to_csv("fashion_beauty_ab_test_assignments.csv", index=False)